# Compsognathus: learn quiet bilateral balance

[Open in Colab](https://colab.research.google.com/github/kuds/mesozoic-labs/blob/codex/compsognathus-foot-research/notebooks/compsognathus_balance_study.ipynb)

Source loads directly from PR527. Mount your existing Drive to inspect saved Compsognathus and T. rex experiments and replay the selected compatible Compsognathus checkpoint. **No source ZIP or model upload is needed.** Existing run files are read in place; new reports and training outputs go to a separate study folder.

The default `baseline` mode imports saved reports and evaluates the existing model. For the matched learning study, A = current reward/filter off; B = current reward/10 Hz; C = bilateral reward/filter off; D = bilateral reward/10 Hz. Use policy seeds 42, 43 and 44 for each arm. Choose `probe` first to check new training, then `full` for 11M steps.

Old checkpoints provide measured baselines. The four new training arms start fresh so the comparison isolates the declared changes. Existing artifacts keep their original identities; a failed compatibility check is reported rather than bypassed.


In [ ]:
# @title Choose existing references or a new study run
MODE = "baseline"  # @param ["baseline", "probe", "full"]
ARM = "D"  # @param ["A", "B", "C", "D"]
TRAINING_SEED = 42  # @param [42, 43, 44] {type:"raw"}
STUDY_NAME = "compsognathus-balance-v1"  # @param {type:"string"}
COMPY_REFERENCE_RUN = "20260909_162812"  # @param {type:"string"}
TREX_REFERENCE_RUN = "20260821_142144"  # @param {type:"string"}
REPLAY_EXISTING_MODEL = True  # @param {type:"boolean"}
PROBE_UPDATES = 2

In [ ]:
# @title Mount existing Drive and load source directly from the PR
import json
import os
import re
import subprocess
import sys
import uuid
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive did not mount; persistent storage is required")
BASE = Path("/content/drive/MyDrive/mesozoic-labs")
if not STUDY_NAME or STUDY_NAME in {".", ".."} or Path(STUDY_NAME).name != STUDY_NAME:
    raise ValueError("STUDY_NAME must be one folder name")
STUDY = BASE / STUDY_NAME
STUDY.mkdir(parents=True, exist_ok=True)
token = str(uuid.uuid4())
storage_check = STUDY / (".write-check-" + token)
try:
    with storage_check.open("w") as stream:
        stream.write(token)
        stream.flush()
        os.fsync(stream.fileno())
    if storage_check.read_text() != token:
        raise IOError("Drive write verification failed")
finally:
    storage_check.unlink(missing_ok=True)

EXPECTED_IMPLEMENTATION = "sha256:42d654eea30bd6c4816d1b54f512062761a34ea4862bfcfb96861ff548f3d75f"
saved_plan = STUDY / "study_plan.json"
saved = json.loads(saved_plan.read_text()) if saved_plan.exists() else None
revision = saved["source_commit"] if saved else "refs/pull/527/head"
if saved and (not re.fullmatch(r"[0-9a-f]{40}", revision) or saved["implementation_sha256"] != EXPECTED_IMPLEMENTATION):
    raise RuntimeError("Open the notebook snapshot saved with this study, or use a new study name for changed source")

REPO = Path("/content/mesozoic-balance-pr527")
REMOTE = "https://github.com/kuds/mesozoic-labs.git"


def git(*args):
    return subprocess.check_output(["git", "-C", str(REPO), *args], text=True).strip()


if not (REPO / ".git").exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError("Source directory contains other files; use a fresh runtime")
    REPO.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["git", "init", "-q", str(REPO)])
    git("remote", "add", "origin", REMOTE)
if git("remote", "get-url", "origin") != REMOTE:
    raise RuntimeError("Source checkout has an unexpected remote")
if git("status", "--porcelain"):
    raise RuntimeError("Source checkout contains changes; use a fresh runtime")
git("fetch", "--quiet", "--depth=1", "origin", revision)
target = git("rev-parse", "FETCH_HEAD")
if any(name == "environments" or name.startswith("environments.") for name in sys.modules):
    if git("rev-parse", "HEAD") != target:
        raise RuntimeError("Study modules are already imported from another revision; use a fresh runtime")
git("checkout", "--quiet", "--detach", target)
os.chdir(REPO)
print("Source revision:", target)
print("Existing experiments:", BASE / "logs")
print("New persistent results:", STUDY)

In [ ]:
# @title Install the training dependencies
import subprocess
import sys

# On a later session, restore the recorded core package versions before import.
# Python itself is supplied by Colab; a changed Python needs a matching runtime.
install = [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[train]"]
saved_plan = STUDY / "study_plan.json"
if saved_plan.exists():
    runtime = json.loads(saved_plan.read_text())["runtime"]
    if sys.version.split()[0] != runtime["python"]:
        raise RuntimeError(
            "This study requires Python "
            + runtime["python"]
            + "; this runtime has "
            + sys.version.split()[0]
            + ". Use a Colab runtime with the recorded Python version. Do not mix results across runtimes."
        )
    constraints = Path("/content/balance-study-constraints.txt")
    constraints.write_text("\n".join(name + "==" + value for name, value in runtime.items() if name != "python") + "\n")
    install.extend(["--constraint", str(constraints)])
subprocess.check_call(install)
sys.path.insert(0, str(REPO))
from environments.compsognathus.experiments.balance_identity import study_source_fingerprint

if study_source_fingerprint() != EXPECTED_IMPLEMENTATION:
    raise ValueError("Installed study source does not match this notebook")

In [ ]:
# @title Prepare the study and calibrate the home reference once
from environments.compsognathus.scripts.train_balance_study import prepare_study

plan = prepare_study(STUDY)
print("Arms:", list(plan["arms"]))
print("Training seeds:", plan["training_seeds"])
print("Proposed physical targets:", plan["behavior_targets"])
print("Package versions:", plan["runtime"])
if not (STUDY / "calibration.json").exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "environments.compsognathus.scripts.train_balance_study",
            "calibrate",
            "--output",
            str(STUDY),
        ]
    )
calibration = json.loads((STUDY / "calibration.json").read_text())
home = calibration["home_reference"]
print("Home reference complete episodes:", sum(row["full_horizon"] for row in home), "/", len(home))

# Keep the exact notebook beside its frozen study plan for later sessions.
snapshot = STUDY / "compsognathus_balance_study.ipynb"
if not snapshot.exists():
    snapshot.write_bytes((REPO / "notebooks/compsognathus_balance_study.ipynb").read_bytes())

In [ ]:
# @title Reuse saved Drive experiments and their selected models
from environments.compsognathus.experiments.balance_drive import (
    discover_balance_runs,
    evaluate_saved_baseline,
    save_historical_baselines,
)
from environments.shared.plant_contract import PlantContractError

historical_runs = discover_balance_runs(BASE)
save_historical_baselines(historical_runs, STUDY / "references" / "historical_baselines.json")
print("Discovered saved stance runs:", len(historical_runs))
preferred = {"compsognathus": COMPY_REFERENCE_RUN, "trex": TREX_REFERENCE_RUN}
selected_references = {}
for species, run_id in preferred.items():
    matches = [row for row in historical_runs if row["species"] == species and row["run_id"] == run_id]
    if not matches:
        print(
            "Reference not found:",
            species,
            run_id,
            "— available:",
            [row["run_id"] for row in historical_runs if row["species"] == species],
        )
        continue
    selected_references[species] = matches[0]
    print(species, run_id, matches[0]["status"], matches[0].get("saved_metrics", {}))
    print("Saved selected pair:", matches[0]["checkpoint_pair"])

reference = selected_references.get("compsognathus")
pair = reference["checkpoint_pair"] if reference else None
if REPLAY_EXISTING_MODEL and pair:
    n_replay = 4 if MODE == "probe" else 40
    replay_output = STUDY / "references" / "replays" / (COMPY_REFERENCE_RUN + "_n" + str(n_replay))
    if (replay_output / "baseline.json").exists():
        baseline = json.loads((replay_output / "baseline.json").read_text())
        if baseline["pair"] != pair:
            raise ValueError("Saved Drive model/config changed since replay; use a new study folder")
        print("Reusing completed baseline replay:", baseline["summary"])
    elif replay_output.exists():
        print("A previous replay did not complete; retained evidence:", replay_output)
    else:
        try:
            baseline = evaluate_saved_baseline(pair, replay_output, seeds=tuple(range(11042, 11042 + n_replay)))
            print("Existing Compy policy, measured with new balance metrics:", baseline["summary"])
        except PlantContractError as exc:
            print("Saved model replay blocked by compatibility verification:", str(exc))
            print("The existing reports remain available. No checkpoint identity was changed.")
else:
    print("Saved reports imported. Model replay disabled or an explicitly recorded pair is unavailable.")
print(
    "Historical report support and new physics-level load metrics have different definitions; compare labels carefully."
)

## Run the selected arm and seed

A full run preserves the existing four-environment PPO recipe and 11M budget, with screening every 250k steps. Each checkpoint stores its matching normalization file and hashes. Only screening chooses a winner; confirmation uses 40 separate seeds once afterward. Training uses CPU for this small policy, even if Colab provides a GPU.

Run A–D for seed 42, then repeat for seeds 43 and 44. The cell refuses to overwrite an existing run. Short probes have separate folders. This first version starts fresh runs; it does not resume interrupted runs or initialize from old canonical checkpoints. If a runtime ends early, retain its files and use a new study directory for a fresh replacement, documenting the interruption. Do not combine incomplete or differently configured studies as successful replications.


In [ ]:
# @title Run the new training arm when selected
from environments.compsognathus.scripts.train_balance_study import train_balance_arm

if MODE == "baseline":
    print("Baseline inspection completed; no new policy training requested.")
else:
    options = {"probe_updates": PROBE_UPDATES, "evaluation_episodes": 4} if MODE == "probe" else {}
    result = train_balance_arm(STUDY, ARM, int(TRAINING_SEED), **options)
    print(json.dumps(result, indent=2))

In [ ]:
# @title Compare all declared full runs
from environments.compsognathus.scripts.train_balance_study import summarize_study

comparison = summarize_study(STUDY)
for row in comparison["runs"]:
    print(row["arm"], row["seed"], row["status"], "balance qualified:", row["learned_balance_qualified"])
print("Reports:", STUDY / "comparison.json", "and", STUDY / "comparison.csv")

## Read the results

`run_summary.json` records whether a completed full run met the study behavior targets. `confirmation.json` contains all confirmation episodes, the unchanged stance criteria projected onto the canonical reward, and the additional balance criteria. `screening_history.json` shows how checkpoint selection progressed. `updates.json` records learning rate, exploration coefficient, learned action standard deviation, and optimizer diagnostics. `selected_trace.csv` contains 50 Hz control-boundary traces; physical balance aggregates sample every 2 ms physics step.

Compare physical behavior across reward variants, not raw training return. A pass here is a research result; disturbance recovery, deliberate foot unloading, and production promotion remain separate validation steps.


In [ ]:
# @title Finish this session and flush pending Drive uploads
FINISH_SESSION = True  # @param {type:"boolean"}

# Leave enabled for a one-run session; disable if more arms will run here.
# Google Colab flushes outstanding writes and then unmounts Drive.
# Remount using the setup cell before running another arm in this runtime.
if FINISH_SESSION:
    drive.flush_and_unmount(timeout_ms=300000)
    print("Drive writes flushed and Drive unmounted. This session can be disconnected.")
else:
    print("Results are saved under", STUDY)
    print("When finished, enable FINISH_SESSION and run this cell to flush pending Drive uploads.")